In [ ]:
import kagglehub
import boto3
import os

# download dataset
path = kagglehub.dataset_download("shivamb/netflix-shows")

print(path)

Kaggle to Amazon S3 to parquet file 

In [ ]:
import kagglehub
import boto3
import os

# Download dataset
path = kagglehub.dataset_download("shivamb/netflix-shows")

print("Dataset downloaded to:", path)

# Locate CSV file
csv_file = os.path.join(path, "netflix_titles.csv")

print("CSV File:", csv_file)

# Create S3 client
s3 = boto3.client("s3")

# Your bucket name
bucket_name = "netflix-validation"

# Upload file to S3
s3.upload_file(
     csv_file,
     bucket_name,
     "Raw-files/netflix_titles.csv"
 )

# print("File uploaded successfully!")

In [ ]:
import pandas as pd
df = pd.read_csv(path + "/netflix_titles.csv")
df.head()
df.drop_duplicates(inplace=True)

In [ ]:
print("Shape of the data set")
print(df.shape)

print("\nColumns present in the data set")
print(df.columns)

print("\nInformation of the data set")
df.info()

In [ ]:
df.to_parquet("netflix.parquet")

Snowflake connector

In [ ]:
import snowflake.connector

conn = snowflake.connector.connect(
    user='t****t',
    password='t****x',
    account='ve57892.ap-southeast-7.aws'
)

print("Connected!")

In [ ]:
cursor = conn.cursor()

cursor.execute("CREATE DATABASE IF NOT EXISTS netflix_db")
cursor.execute("USE DATABASE netflix_db")

cursor.execute("CREATE SCHEMA IF NOT EXISTS raw")

print("Database and schema created!")

In [ ]:
cursor.execute("""
CREATE WAREHOUSE IF NOT EXISTS netflix_wh
WITH WAREHOUSE_SIZE='XSMALL'
AUTO_SUSPEND = 60
AUTO_RESUME = TRUE
""")

cursor.execute("USE WAREHOUSE netflix_wh")

print("Warehouse ready!")

In [ ]:
import kagglehub
import os

# Download dataset
path = kagglehub.dataset_download("obaidhere/netflix-content-strategy-genre-and-rating-analysis")

print("Dataset folder path:", path)
print("Files inside folder:", os.listdir(path))

In [ ]:
inport pandas as pd
df = pd.read_csv(path + "/netflix_titles.csv")
df.head()df = pd.read_csv(path + "/netflix_titles.csv")
df.head()

In [ ]:
from snowflake.connector.pandas_tools import write_pandas

# Select database/schema first
cursor.execute("USE DATABASE netflix_db")
cursor.execute("USE SCHEMA raw")

success, nchunks, nrows, _ = write_pandas(
    conn=conn,
    df=df,
    table_name='NETFLIX_RAW',
    auto_create_table=True
)

print(success)
print(nrows)